# Planners-11: Unified Planning

[Version Python](Planners-11-Unified-Planning-Python.ipynb) · [Parcours de la partie](README.md)

**Navigation** : [Index](../../README.md) | [<< LLM Planning](Planners-10-LLM-Planning-Python.ipynb) | [LOOP >>](Planners-12-LOOP-Python.ipynb)

## Interface Unifiee pour Multi-Solveurs

Ce notebook presente **unified-planning**, une bibliotheque Python qui offre une interface unique pour interagir avec de multiples planificateurs (Fast Downward, Pyperplan, Tamer, etc.). Cette abstraction permet de comparer facilement différents solveurs et de changer de planificateur sans reecrire le code.

**Environnement du jumeau** — C# et Python 3.13 avec unified-planning 1.3.0, up-pyperplan 1.1.0 et up-fast-downward 1.0.0.


### Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. Utiliser l'API unified-planning pour définir des problèmes de planification
2. Comparer les performances de différents solveurs sur un même problème
3. Exploiter les fonctionnalites avancees : planification temporelle, fluents numériques
4. Convertir un modèle Unified Planning vers et depuis PDDL
5. Choisir le solveur adapte selon le type de problème
6. Integrer unified-planning dans un pipeline de planification

### Prerequis
- Avoir suivi [Planners-02-PDDL-Basics-Python](../01-Foundation/Planners-02-PDDL-Basics-Python.ipynb)
- Connaissance de C# orienté objet
- Comprendre le modèle STRIPS (prédicats, actions, préconditions, effets)

### Durée estimee : 40 minutes

***

## 1. Introduction a unified-planning

**unified-planning** est un projet de l'Union Europeenne (AIPlan4EU) qui vise a democratise l'acces aux outils de planification. Elle offre :

- Une API de construction de modèles accessible depuis C# via Python.NET.NET pour définir domaines et problèmes
- Une interface unifiee pour 10+ planificateurs
- Support pour PDDL, planification temporelle, et contraintes numériques
- Comparaison automatique des performances

### 1.1 Planificateurs supportes

| Planificateur | Type | Optimalite | Spécialités |
|---------------|------|------------|--------------|
| **Fast Downward** | Classique | Optimal/Satisficing | Heuristiques LM-cut, FF |
| **Pyperplan** | Classique | Satisficing | Leger, rapide |
| **Tamer** | Classique | Satisficing | Planification avec préférences |
| **ENHSP** | Numérique | Optimal | Fluents numériques, temps |
| **UP-SMT** | SMT | Optimal | Contraintes complexes |
| **LPG** | Temporel | Satisficing | PDDL 2.1, durées |
| **OSP** | Oversubscription | Partiel | Objectifs multiples |

### 1.2 Installation et configuration

La bibliotheque s'installe avec pip. Chaque planificateur a son propre package optionnel.

In [1]:
#nullable enable
#r "nuget: pythonnet, 3.1.0"
#load "../../../Config/PythonNetLoader.cs"

using System;
using System.Diagnostics;
using System.Collections.Generic;
using System.Globalization;
using System.IO;
using System.Linq;
using Python.Runtime;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;

// Le modèle est construit par C#. Python.NET donne accès aux vrais objets et moteurs Unified Planning.
var runtime = PythonNetLoader.Initialize(typeof(PythonEngine).Assembly, "unified_planning");
if (runtime is not null)
    PythonEngine.BeginAllowThreads();
dynamic up = null!;
dynamic pddl = null!;
dynamic upEngines = null!;
dynamic plannerFactory = null!;
using (Py.GIL())
{
    up = Py.Import("unified_planning.shortcuts");
    pddl = Py.Import("unified_planning.io");
    upEngines = Py.Import("unified_planning.engines");
    using var moduleScope = Py.CreateScope();
    moduleScope.Exec("""
    from unified_planning.shortcuts import get_environment, OneshotPlanner
    import warnings
    get_environment().credits_stream = None
    warnings.filterwarnings("ignore", message=r"We cannot establish whether")
    warnings.filterwarnings("ignore", message=r"Pyperplan does not support timeout.")

    def build_planner(name):
        return OneshotPlanner(name=name)
    """);
    // UP 1.3 inspecte la pile Python de la fabrique même quand les crédits sont masqués.
    // Cet appel ciblé ajoute le cadre Python attendu par OneshotPlanner.
    plannerFactory = moduleScope.Get("build_planner");
    using var package = Py.Import("unified_planning");
    Console.WriteLine($"Unified Planning {package.GetAttr("__version__")} | Python.NET {typeof(Py).Assembly.GetName().Version}");
}
Console.WriteLine("Le moteur Pyperplan ne prend pas en charge le délai demandé par l’interface. Fast Downward reçoit ce délai.");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages pythonnet, 3.1.0

Unified Planning 1.3.0 | Python.NET 3.1.0.0


Le moteur Pyperplan ne prend pas en charge le délai demandé par l’interface. Fast Downward reçoit ce délai.


Verification des moteurs de planification disponibles sur le système et affichage de leurs capacites (planification classique, temporelle, numérique, etc.).

In [2]:
#nullable enable
using (Py.GIL())
{
    // La factory expose les moteurs réellement importables dans cet environnement.
    dynamic engines = upEngines.Factory(up.get_environment()).engines;
    int count = (int)((PyObject)engines).Length();
    var entries = Enumerable.Range(0, count).Select(i => (string)engines[i].ToString())
        .OrderBy(name => name, StringComparer.Ordinal).ToArray();
    foreach (string name in entries.Where(n => n.Contains("planner") || n.Contains("oneshot")
        || n.StartsWith("fast-downward") || n is "tamer" or "pyperplan"))
        Console.WriteLine($"  {name}");
    Console.WriteLine($"{count} moteurs détectés dans la factory.");
    if (!new[] { "pyperplan", "fast-downward", "fast-downward-opt" }.All(entries.Contains))
        throw new InvalidOperationException("Les deux plugins et le moteur d’optimisation sont requis.");
}

  fast-downward


  fast-downward-grounder


  fast-downward-opt


  fast-downward-reachability-grounder


  pyperplan


  replanner[fast-downward-opt]


  replanner[fast-downward]


  replanner[pyperplan-opt]


  replanner[pyperplan]


25 moteurs détectés dans la factory.


***

## 2. Definition de Problèmes avec l'API

L'API unified-planning permet de définir des problèmes directement en Python, sans ecrire de fichiers PDDL. C'est souvent plus intuitif et permet l'integration avec d'autres outils Python.

### 2.1 Types et Fluents

Les **types utilisateur** (UserType) representent les objets du domaine. Les **fluents** sont des fonctions qui retournent une valeur selon l'état du monde.

In [3]:
#nullable enable
dynamic locationType;
dynamic robotType;
dynamic robotAt;
dynamic connected;
dynamic visited;
using (Py.GIL())
{
    locationType = up.UserType("Location");
    robotType = up.UserType("Robot");
    // Les signatures définissent les objets acceptés par chaque relation.
    robotAt = up.Fluent("robot_at", up.BoolType(), r: robotType, l: locationType);
    connected = up.Fluent("connected", up.BoolType(), l1: locationType, l2: locationType);
    visited = up.Fluent("visited", up.BoolType(), l: locationType);
    Console.WriteLine($"Types : {locationType}, {robotType}");
    Console.WriteLine($"Fluents : {robotAt}, {connected}, {visited}");
}

Types : Location, Robot


Fluents : bool robot_at[r=Robot, l=Location], bool connected[l1=Location, l2=Location], bool visited[l=Location]


### 2.2 Actions et Effets

Une **action** a des paramètres, des préconditions et des effets. L'API utilise des méthodes fluentes pour ajouter ces éléments.

In [4]:
#nullable enable
dynamic move;
using (Py.GIL())
{
    move = up.InstantaneousAction("move", r: robotType, l_from: locationType, l_to: locationType);
    dynamic robotParam = move.parameter("r");
    dynamic from = move.parameter("l_from");
    dynamic to = move.parameter("l_to");
    // Une arête et la position courante autorisent le déplacement.
    move.add_precondition(connected(from, to));
    move.add_precondition(robotAt(robotParam, from));
    move.add_effect(robotAt(robotParam, from), false);
    move.add_effect(robotAt(robotParam, to), true);
    move.add_effect(visited(to), true);
    Console.WriteLine(move.ToString());
}

action move(Robot r, Location l_from, Location l_to) {
    preconditions = [
      connected(l_from, l_to)
      robot_at(r, l_from)
    ]
    effects = [
      robot_at(r, l_from) := false
      robot_at(r, l_to) := true
      visited(l_to) := true
    ]
  }


### 2.3 Problème complet

Un problème complet comprend : le domaine (types, fluents, actions) et l'instance (objets, état initial, buts).

In [5]:
#nullable enable
dynamic navigation;
// Les branches symétriques contiennent le détour vers L13 et sept impasses.
var corridors = new[] { (0, 1), (1, 2), (2, 4), (4, 7), (7, 10), (10, 13), (10, 11), (13, 11) };
var deadEnds = new[] { (1, 3), (4, 5), (2, 6), (7, 8), (2, 9), (10, 12), (13, 14) };
var edges = corridors.Concat(deadEnds).SelectMany(edge => new[] { edge, (edge.Item2, edge.Item1) }).ToArray();
using (Py.GIL())
{
    navigation = up.Problem("robot_navigation");
    foreach (dynamic fluent in new[] { robotAt, connected, visited })
        navigation.add_fluent(fluent, default_initial_value: false);
    navigation.add_action(move);
    var locations = Enumerable.Range(0, 15).Select(i => (dynamic)up.Object($"L{i}", locationType)).ToArray();
    dynamic robot = up.Object("robot1", robotType);
    foreach (dynamic location in locations) navigation.add_object(location);
    navigation.add_object(robot);
    foreach (var (from, to) in edges)
        navigation.set_initial_value(connected(locations[from], locations[to]), true);
    navigation.set_initial_value(robotAt(robot, locations[0]), true);
    navigation.set_initial_value(visited(locations[0]), true);
    navigation.add_goal(robotAt(robot, locations[11]));
    Console.WriteLine($"{navigation.name} | {locations.Length} lieux | {edges.Length} arêtes | {deadEnds.Length} impasses.");
    Console.WriteLine("Le robot part de L0 et vise L11. Deux branches relient le corridor au but.");
}

robot_navigation | 15 lieux | 30 arêtes | 7 impasses.


Le robot part de L0 et vise L11. Deux branches relient le corridor au but.


### Interpretation : API unified-planning

**Structure du modèle** : Le problème suit le paradigme STRIPS avec une API orientée objet appelée depuis C#.

| Élément | API unified-planning | Équivalent PDDL |
|---------|---------------------|-----------------|
| Types | `UserType('Location')` | `(:types location)` |
| Fluents | `Fluent('robot_at', BoolType(), ...)` | `(:predicates (robot_at ?r ?l))` |
| Actions | `InstantaneousAction('move', ...)` | `(:action move ...)` |
| Préconditions | `action.add_precondition(...)` | `:precondition (and ...)` |
| Effets | `action.add_effect(...)` | `:effect (and ...)` |

**Points clés** :
1. L'API est plus verbeuse que PDDL mais plus expressive en Python
2. Les paramètres d'action sont accessibles via `action.parameter('name')`
3. `default_initial_value` simplifie l'initialisation des fluents

### Extension guidée — Ramener deux caisses au dépôt

Réutilisez les types et les positions du robot. Ajoutez un type `Box` ainsi que les fluents `free`, `holding` et `at_box`. La caisse et le robot doivent partager un lieu au moment du chargement.

```csharp
using (Py.GIL())
{
    dynamic boxType = up.UserType("Box");
    dynamic free = up.Fluent("free", up.BoolType(), robot: robotType);
    dynamic holding = up.Fluent("holding", up.BoolType(), robot: robotType, box: boxType);
    dynamic atBox = up.Fluent("at_box", up.BoolType(), box: boxType, location: locationType);
    dynamic load = up.InstantaneousAction("charger", r: robotType, b: boxType, l: locationType);
    dynamic r = load.parameter("r");
    dynamic b = load.parameter("b");
    dynamic l = load.parameter("l");
    load.add_precondition(robotAt(r, l));
    load.add_precondition(atBox(b, l));
    load.add_precondition(free(r));
    load.add_effect(atBox(b, l), false);
    load.add_effect(free(r), false);
    load.add_effect(holding(r, b), true);
}
```

Construisez un domaine avec le déplacement, le chargement et le déchargement. Créez quatre lieux, un robot et deux caisses. Le robot part du dépôt et les caisses attendent en zone A et en zone B. Le but place les deux caisses au dépôt. Le déchargement rend le robot libre et place au sol la caisse tenue. Dans ce modèle, `free(r)` interdit de charger une deuxième caisse.

***

## 3. Resolution et Comparaison de Solveurs

L'un des grands avantages d'unified-planning est la possibilite de tester facilement différents solveurs sur le même problème.

In [6]:
#nullable enable
record PlanningRun(string Engine, string Status, double Seconds, PyObject? Plan, string[] Actions)
{
    public bool Solved => Status is "SOLVED_SATISFICING" or "SOLVED_OPTIMALLY";
}

PlanningRun Solve(PyObject problem, string name, int? timeout = null)
{
    using (Py.GIL())
    {
        var timer = Stopwatch.StartNew();
        dynamic solver = plannerFactory(name);
        try
        {
            // La fabrique sélectionne le moteur par nom. Le statut est lu sur son résultat.
            dynamic outcome = timeout is int seconds ? solver.solve(problem, timeout: seconds) : solver.solve(problem);
            string status = outcome.status.name.ToString();
            var plan = (PyObject)outcome.plan;
            if (plan is null || plan.IsNone())
                return new(name, status, timer.Elapsed.TotalSeconds, null, Array.Empty<string>());
            dynamic steps = plan.GetAttr("actions");
            int length = (int)((PyObject)steps).Length();
            var text = Enumerable.Range(0, length).Select(i => (string)steps[i].ToString()).ToArray();
            return new(name, status, timer.Elapsed.TotalSeconds, plan, text);
        }
        finally
        {
            solver.__exit__(null, null, null);
        }
    }
}
void ShowPlan(PlanningRun run)
{
    Console.WriteLine($"{run.Engine} | {run.Status} | {run.Seconds:F3} s | {run.Actions.Length} actions");
    foreach (var (action, index) in run.Actions.Select((step, i) => (step, i + 1)))
        Console.WriteLine($"  {index}. {action}");
}

var first = Solve((PyObject)navigation, "pyperplan");
ShowPlan(first);
if (!first.Solved || first.Actions.Length != 6)
    throw new InvalidOperationException("Le parcours Pyperplan doit atteindre L11 en six actions dans ce graphe.");

pyperplan | SOLVED_SATISFICING | 0.034 s | 6 actions


  1. move(robot1, L0, L1)


  2. move(robot1, L1, L2)


  3. move(robot1, L2, L4)


  4. move(robot1, L4, L7)


  5. move(robot1, L7, L10)


  6. move(robot1, L10, L11)


### Interpretation : Resolution avec un seul solveur

**Sortie obtenue** : pyperplan resout le labyrinthe et renvoie un statut `SOLVED_SATISFICING` avec un plan de **6 actions** menant le robot de L0 (entree) a L11 (sortie).

**Le plan trouvé suit le chemin court.** Il traverse le corridor principal L0 -> L1 -> L2 -> L4 -> L7 -> L10, puis atteint L11. Les sept impasses (L3, L5, L6, L8, L9, L12, L14) sont absentes du plan. Les solveurs peuvent exploiter une heuristique pour guider leur exploration ; cette sortie donne la trace des actions retenues.

**Workflow de resolution** :
1. Créer le solveur (`OneshotPlanner`)
2. Appeler `solve(problem)`
3. Analyser le statut de retour
4. Extraire le plan si succes

**Statuts possibles** :
- `SOLVED_SATISFICING` : Plan trouve (pas forcement optimal)
- `SOLVED_OPTIMALLY` : Plan optimal trouve
- `UNSOLVABLE_PROVEN` signifie qu'une preuve d'insolubilité est fournie.
- `UNSOLVABLE_INCOMPLETELY` indique que la recherche n'a pas abouti à une preuve.
- `TIMEOUT` : Timeout atteint
- `INTERNAL_ERROR` désigne un incident interne rapporté par le moteur.

Dans `Solve`, le bloc `finally` appelle `__exit__` pour rendre les ressources du moteur. L'adaptateur d'appel garde la pile Python requise par Unified Planning 1.3 pour traiter les crédits. Le statut et les six actions ci-dessus proviennent du moteur Pyperplan.

### 3.1 Comparaison multi-solveurs

Comparons les performances de plusieurs planificateurs sur le même problème.

In [7]:
#nullable enable
// On garde le même objet Problem et on rapproche les statuts, actions et temps observés.
var enginesToCompare = new[] { "pyperplan", "fast-downward" };
var comparison = enginesToCompare.Select(engine => Solve((PyObject)navigation, engine, 30)).ToArray();
foreach (var run in comparison)
{
    Console.WriteLine($"{run.Engine,-16} | {run.Status,-20} | {run.Seconds,7:F3}s | {run.Actions.Length} actions");
    if (!run.Solved || run.Actions.Length != 6)
        throw new InvalidOperationException("Le contrôle du graphe de navigation exige les deux plans valides de six actions.");
}

pyperplan        | SOLVED_SATISFICING   |   0.012s | 6 actions


fast-downward    | SOLVED_SATISFICING   |   0.183s | 6 actions


### Interpretation : Comparaison multi-solveurs

**Sortie obtenue** : Les deux solveurs testes resolvent le labyrinthe, avec des temps différents mais un plan de même longueur (6 actions).

| Solveur | Statut | Plan |
|---------|--------|------|
| pyperplan | `SOLVED_SATISFICING` | 6 actions |
| fast-downward | `SOLVED_SATISFICING` | 6 actions |

**Pourquoi pyperplan est plus rapide ici** : sur un labyrinthe de 15 noeuds, l'espace de recherche est petit, donc le temps est domine par le **démarrage du moteur** (startup overhead), pas par la recherche elle-même. pyperplan est un planner STRIPS leger ecrit en Python pur : il se lance presque instantanement. fast-downward est un moteur C++ externe, beaucoup plus puissant (heuristiques configurables : LM-cut, FF, merge-and-shrink) mais avec une phase d'initialisation plus lourde. Le constat "pyperplan plus rapide" reflete donc la légèreté du moteur sur un petit problème, **pas** une inferiorite de fast-downward -- dont les heuristiques prennent tout leur sens sur des problèmes plus gros (voir ci-dessous).

**Workflow de comparaison** :

1. **Initialisation** : Créer le problème une seule fois
2. **Tests séquentiels** : Lancer chaque solveur avec un timeout
3. **Collecte de résultats** : Statut, temps, longueur du plan
4. **Analyse** : Identifier le plus rapide, le plan le plus court

> **Note technique** : unified-planning permet de changer de solveur en modifiant une seule ligne (le nom `fast-downward` au lieu de `pyperplan` dans `Solve`), ce qui facilite enormement le benchmarking. Les deux plans suivent le chemin court (L0-...-L10-L11) en 6 actions ; sur un problème plus riche ou un solveur serait satisficing et l'autre optimal, les longueurs de plan pourraient diverger.

Les temps exacts sont ceux **imprimes par la cellule de comparaison ci-dessus**. Ils dependent de la machine et de son démarrage a froid : les recopier ici les figerait sur une exécution alors que la cellule les reimprime a chaque passage — c'est pourquoi la colonne `Temps` n'est pas reproduite dans ce tableau. Ce qui reste stable d'une exécution a l'autre, et qui porte la lecture, c'est l'**ordre** des solveurs et la longueur des plans.

Affichage formate sous forme de tableau des résultats de la comparaison entre solveurs, incluant le statut, le temps de resolution et la longueur du plan produit.

In [8]:
#nullable enable
void ComparisonTable(IReadOnlyList<PlanningRun> rows)
{
    Console.WriteLine("{0,-16} | {1,-20} | {2,9} | {3,7}", "Solveur", "Statut", "Temps (s)", "Actions");
    foreach (var run in rows)
        Console.WriteLine("{0,-16} | {1,-20} | {2,9:F3} | {3,7}",
            run.Engine, run.Status, run.Seconds, run.Actions.Length);
    // Les critères de temps et de longueur sont distincts, avec les égalités conservées.
    var solved = rows.Where(run => run.Solved).ToArray();
    if (solved.Length == 0)
        return;
    var fastest = solved.MinBy(run => run.Seconds)!;
    int shortest = solved.Min(run => run.Actions.Length);
    Console.WriteLine($"Plus rapide sur ce run : {fastest.Engine} ({fastest.Seconds:F3}s).");
    Console.WriteLine($"Plans les plus courts : {string.Join(", ", solved.Where(run => run.Actions.Length == shortest).Select(run => run.Engine))} ({shortest} actions).");
}
ComparisonTable(comparison);

Solveur          | Statut               | Temps (s) | Actions


pyperplan        | SOLVED_SATISFICING   |     0.012 |       6


fast-downward    | SOLVED_SATISFICING   |     0.183 |       6


Plus rapide sur ce run : pyperplan (0.012s).


Plans les plus courts : pyperplan, fast-downward (6 actions).


### Interprétation : qui gagne, et quand ?

**Sortie obtenue** : sur ce labyrinthe de 15 nœuds, **pyperplan est le plus rapide** —
l'écart vient du démarrage du moteur, que la cellule de comparaison imprime à chaque
exécution — mais les deux solveurs produisent un **plan de 6 actions**.
Il n'y a pas de « vainqueur » absolu : chaque moteur a son domaine de prédilection.

**Lecture des colonnes du tableau** (celui qu'imprime la cellule ci-dessus) :

- **Temps** : latence de résolution, mesure de machine — dominée sur cette instance par
  le démarrage du moteur, pas par la recherche. C'est pourquoi elle n'est pas recopiée
  dans les cellules d'interprétation.
- **Statut** : `SOLVED_OPTIMALLY` (plan optimal prouvé), `SOLVED_SATISFICING` (plan
  trouvé, optimalité non garantie), `UNSOLVABLE_PROVEN` (insolubilité prouvée),
  `UNSOLVABLE_INCOMPLETELY` (recherche sans preuve) ou `INTERNAL_ERROR` (incident interne).
- **Longueur Plan** : nombre d'actions du plan, plus court étant mieux. Les deux moteurs
  convergent ici vers le chemin optimal de 6 moves.

**Comparaison théorique des solveurs unified-planning** :

| Solveur | Optimalité | Vitesse | Spécialités |
|---------|------------|---------|-------------|
| **Pyperplan** | Satisficing | Très rapide (start léger) | Problèmes simples, prototypage |
| **Fast Downward** | Optimal/Satisficing | Moyenne (start lourd) | Heuristiques puissantes (LM-cut, FF, merge-and-shrink) |
| **Tamer** | Satisficing | Rapide | Préférences, plans avec qualité |
| **ENHSP** | Optimal | Lent | Fluents numériques, PDDL 2.1 |
| **LPG** | Satisficing | Moyenne | Planification temporelle |

**Points clés** :

- Les solveurs **optimaux** (`A*`, recherche informée à heuristique admissible) peuvent certifier l'optimalité du plan.
- Les solveurs **satisficing** (`GBFS`, `EHC`) peuvent rendre
  des plans sous-optimaux.
- Le **même problème** peut donc donner des plans différents selon le solveur choisi.

**Quand utiliser chaque solveur** :
- **Pyperplan** : tests rapides, petits problèmes (< 100 objets) — comme ici, où sa
  légèreté l'emporte.
- **Fast Downward** : production, problèmes complexes — ses heuristiques dominent dès
  que l'espace de recherche devient grand (le startup fixe devient négligeable face au
  coût de recherche).
- **ENHSP** : problèmes numériques (coûts, ressources).
- **LPG** : planification temporelle (PDDL 2.1).

**Leçon de cette instance** : le labyrinthe a 15 nœuds et 7 impasses — assez riche pour
qu'**aucun des deux moteurs ne dégénère** (tous deux trouvent un plan non trivial de
6 actions en évitant les dead-ends), mais assez petit pour que le **temps soit dominé
par le startup**. Pour voir fast-downward prendre l'avantage, il faudrait un graphe
beaucoup plus large, où pyperplan (satisficing, heuristique limitée) ralentirait alors
que fast-downward (heuristiques configurables) resterait efficace.

> **Note importante** : le choix du solveur dépend fortement du type de problème ET de
> sa taille. Un solveur excellent sur Block World peut être médiocre sur Logistics.
> C'est tout l'intérêt d'unified-planning : tester facilement plusieurs solveurs et
> laisser les métriques (temps, longueur de plan) décider.

***

## 4. Fonctionnalites Avancees

unified-planning supporte des extensions au modèle STRIPS classique : planification temporelle et fluents numériques.

### 4.1 Actions duratives (temporelles)

Les **DurativeActions** ont une durée et des effets conditionnes par le temps (au debut, a la fin, pendant).

In [9]:
#nullable enable
dynamic drive;
using (Py.GIL())
{
    dynamic city = up.UserType("City");
    dynamic truck = up.UserType("Truck");
    dynamic package = up.UserType("Package");
    dynamic truckAt = up.Fluent("truck_at", up.BoolType(), t: truck, c: city);
    dynamic packageAt = up.Fluent("package_at", up.BoolType(), p: package, c: city);
    dynamic packageIn = up.Fluent("package_in", up.BoolType(), p: package, t: truck);
    dynamic distance = up.Fluent("distance", up.IntType(), c1: city, c2: city);
    drive = up.DurativeAction("drive", t: truck, c1: city, c2: city);
    dynamic t = drive.parameter("t");
    dynamic from = drive.parameter("c1");
    dynamic to = drive.parameter("c2");
    // Le camion quitte la ville au début du trajet et atteint l'autre ville à la fin.
    drive.set_fixed_duration(10);
    drive.add_condition(up.StartTiming(), truckAt(t, from));
    dynamic equalsExpr = ((PyObject)up).GetAttr("Equals");
    drive.add_condition(up.StartTiming(), up.Not(equalsExpr(from, to)));
    drive.add_effect(up.StartTiming(), truckAt(t, from), false);
    drive.add_effect(up.EndTiming(), truckAt(t, to), true);
    Console.WriteLine(drive.ToString());
    Console.WriteLine($"Fluents prévus pour les colis et la distance : {packageAt}, {packageIn}, {distance}.");
}

durative action drive(Truck t, City c1, City c2) {
    duration = [10, 10]
    conditions = [
      [start]:
        truck_at(t, c1)
        (not (c1 == c2))
    ]
    effects = [
      start:
        truck_at(t, c1) := false:
      end:
        truck_at(t, c2) := true:
    ]
    simulated effects = [
    ]
  }


Fluents prévus pour les colis et la distance : bool package_at[p=Package, c=City], bool package_in[p=Package, t=Truck], integer distance[c1=City, c2=City].


### 4.2 Fluents numériques et coûts

Les fluents peuvent etre de type entier ou flottant, permettant de modéliser des ressources (carburant, argent) et des coûts.

### Exercice 1 — Planification temporelle avec ressources numériques

### Contexte : Gestion d'un chantier de construction

Un chantier a 3 tâches a effectuer : `fondations`, `murs`, `toiture`. Certaines tâches ont des dependances et consomment des ressources.

**Contraintes :**
- `murs` ne peut commencer qu'après la fin de `fondations`
- `toiture` ne peut commencer qu'après la fin de `murs`
- Chaque tâche consume du budget : `fondations` = 3000€, `murs` = 5000€, `toiture` = 2000€
- Budget total disponible : 12000€
- Durées : `fondations` = 5 jours, `murs` = 10 jours, `toiture` = 3 jours

### Objectifs

1. Modéliser ce problème comme un problème de **planification temporelle** avec `unified_planning`
2. Définir un fluent numérique `budget_restant` initialise a 12000
3. Définir 3 actions duratives (`DurativeAction`) avec contraintes de temps et effets numériques
4. Résoudre avec un planificateur temporel (Tamer ou similaire)
5. Afficher le plan avec les temps de debut/fin de chaque action
6. (Bonus) Exporter le problème en PDDL avec `PDDLWriter` et vérifier le fichier genere

> **Indices (API utile) :**

- `DurativeAction` + `set_fixed_duration(...)` pour les actions a durée
- `add_condition(StartTiming(), ...)` / `add_condition(EndTiming(), ...)` pour les conditions temporisees
- `add_decrease_effect(EndTiming(), ...)` pour la consommation de budget
- Sequencez les tâches via des fluents booléens (ex: `fondations_ok`) utilises en précondition

**Indice : Exercice de planification temporelle**

**Approche suggeree** pour modéliser le problème de chantier :

1. **Fluents booléens** pour état des tâches :
   - `fondations_ok`, `murs_ok`, `toiture_ok` indiquent si chaque tâche est terminee
   - Ces fluents servent de préconditions pour les tâches suivantes

2. **Fluent numérique** pour budget :
   - `budget_restant` initialise a 12000
   - Chaque action diminue le budget (effet numérique)

3. **Actions duratives** (`DurativeAction`) :
   - `set_fixed_duration(jours)` pour chaque tâche
   - Conditions au debut (`StartTiming()`) : vérifier budget suffisant
   - Conditions au debut (`StartTiming()`) : vérifier dependances (ex: murs_ok pour toiture)
   - Effets a la fin (`EndTiming()`) : marquer tâche OK, diminuer budget

4. **Resolution** :
   - Utiliser un planificateur supportant le temporel (Tamer, ENHSP)
   - Le plan devrait montrer les temps de debut/fin de chaque tâche

> **Note** : Les dependances entre tâches peuvent se modéliser de deux facons : (1) avec un fluent booléen (fondations_ok=True) ou (2) avec une contrainte de timing (start_murs >= end_fondations). L'approche (1) est plus flexible.

In [10]:
#nullable enable
// Exercice 1 : fondations → murs → toiture avec durées et budget numérique.
PyObject? ConstructionSchedule()
{
    // TODO étudiant : créer les actions duratives, leur ordre, le montant disponible et le but.
    return null;
}
var possibleSchedule = ConstructionSchedule();
Console.WriteLine(possibleSchedule is null ? "Exercice de chantier à compléter." : "Problème de chantier construit.");

Exercice de chantier à compléter.


Le modèle suivant ajoute un fluent numérique. La prochaine cellule déclare sa contrainte de batterie.

In [11]:
#nullable enable
dynamic locType;
dynamic at;
dynamic adjacent;
dynamic battery;
dynamic numericMove;
using (Py.GIL())
{
    locType = up.UserType("Loc");
    at = up.Fluent("at", up.BoolType(), l: locType);
    adjacent = up.Fluent("adjacent", up.BoolType(), l1: locType, l2: locType);
    battery = up.Fluent("battery", up.IntType());
    numericMove = up.InstantaneousAction("move", l_from: locType, l_to: locType);
    dynamic from = numericMove.parameter("l_from");
    dynamic to = numericMove.parameter("l_to");
    // Le coût de l'action intervient ici comme consommation d'un fluent d'état.
    numericMove.add_precondition(at(from));
    numericMove.add_precondition(adjacent(from, to));
    numericMove.add_precondition(up.GE(battery(), 10));
    numericMove.add_effect(at(from), false);
    numericMove.add_effect(at(to), true);
    numericMove.add_decrease_effect(battery(), 10);
    Console.WriteLine(numericMove.ToString());
}

action move(Loc l_from, Loc l_to) {
    preconditions = [
      at(l_from)
      adjacent(l_from, l_to)
      (10 <= battery)
    ]
    effects = [
      at(l_from) := false
      at(l_to) := true
      battery -= 10
    ]
  }


Création du problème numérique avec le robot et sa batterie : initialisation des fluents, definition de l'état initial et specification du but a atteindre.

In [12]:
#nullable enable
dynamic batteryProblem;
using (Py.GIL())
{
    batteryProblem = up.Problem("robot_battery");
    batteryProblem.add_fluent(at, default_initial_value: false);
    batteryProblem.add_fluent(adjacent, default_initial_value: false);
    batteryProblem.add_fluent(battery, default_initial_value: 0);
    batteryProblem.add_action(numericMove);

    // Deux arcs orientés suffisent à caractériser le parcours attendu et son bilan d'énergie.
    dynamic a = up.Object("A", locType);
    dynamic b = up.Object("B", locType);
    dynamic c = up.Object("C", locType);
    foreach (dynamic place in new[] { a, b, c }) batteryProblem.add_object(place);
    batteryProblem.set_initial_value(at(a), true);
    batteryProblem.set_initial_value(battery(), 50);
    batteryProblem.set_initial_value(adjacent(a, b), true);
    batteryProblem.set_initial_value(adjacent(b, c), true);
    batteryProblem.add_goal(at(c));
    Console.WriteLine($"{batteryProblem.name} | batterie initiale 50 | but C à deux mouvements.");
}

robot_battery | batterie initiale 50 | but C à deux mouvements.


### Interpretation : Fonctionnalites avancees

**Extensions supportees** :

| Feature | API | Solveurs compatibles |
|---------|-----|---------------------|
| Durée fixe | `set_fixed_duration(n)` | LPG, ENHSP, Temporal planner |
| Durée variable | `set_duration_constraint(...)` | Planificateurs temporels |
| Effets temporises | `StartTiming()`, `EndTiming()` | Planificateurs temporels compatibles |
| Fluents numériques | `IntType()`, `RealType()` | ENHSP, SMT planners |
| Augmentation | `add_increase_effect(...)` | Solveurs numériques |
| Diminution | `add_decrease_effect(...)` | Solveurs numériques |

**Points clés** :
1. Les actions duratives permettent le parallélisme (plusieurs actions en même temps)
2. Les fluents numériques ajoutent un niveau de complexité (solveurs spécifiques necessaires)
3. Toujours vérifier que le solveur supporte les fonctionnalitees utilisees

### Exemple guide 1 : Resolution complete du problème robot_battery

Le problème `robot_battery` est maintenant défini avec un robot qui doit aller de A a C en passant par B, avec une contrainte de batterie (50 unites, 10 par mouvement). Verifions que le problème est coherent et tentons de le résoudre avec Fast Downward (moteur classique qui ne supporte pas les fluents numériques — le plan affiché est le résultat attendu, calculé à la main).

In [13]:
#nullable enable
using (Py.GIL())
{
    // Le PDDL généré révèle la présence du fluent numérique et le seuil dans la précondition.
    string generated = pddl.PDDLWriter(batteryProblem).get_domain().ToString();
    Console.WriteLine(generated[..Math.Min(generated.Length, 400)]);
}
var numericRun = Solve((PyObject)batteryProblem, "fast-downward");
ShowPlan(numericRun);
if (numericRun.Solved)
    Console.WriteLine($"Batterie restante d'après le plan : {50 - 10 * numericRun.Actions.Length}.");
else
{
    Console.WriteLine("Témoin déduit à la main. La sortie du moteur ci-dessus porte son statut effectif.");
    var route = new[] { ("A", "B"), ("B", "C") };
    int charge = 50;
    foreach (var (from, to) in route)
    {
        Console.WriteLine($"move({from}, {to}) | batterie {charge} → {charge - 10}");
        charge -= 10;
    }
}

(define (domain robot_battery-domain)
 (:requirements :strips :typing :numeric-fluents)
 (:types loc)
 (:predicates 
             (at ?l - loc)
             (adjacent ?l1 - loc ?l2 - loc)
 )
 (:functions 
             (battery)
 )
 (:action move
  :parameters ( ?l_from - loc ?l_to - loc)
  :precondition (and (at ?l_from) (adjacent ?l_from ?l_to) (<= 10 (battery)))
  :effect (and (not (at ?l_from))


fast-downward | INTERNAL_ERROR | 0.455 s | 0 actions


Témoin déduit à la main. La sortie du moteur ci-dessus porte son statut effectif.


move(A, B) | batterie 50 → 40


move(B, C) | batterie 40 → 30


### Interpretation : Exemple guide 1

**Cycle complet** : definition du problème (fluents booléens + numériques) → tentative avec Fast Downward (échoue : INTERNAL_ERROR, fluents numériques non supportés par le moteur classique) → plan attendu calculé à la main.

**Points a retenir** :
1. Les fluents numériques (`IntType`) permettent de modéliser des **ressources consommables** (batterie, carburant, budget)
2. La précondition `GE(battery(), 10)` empeche toute action si la ressource est insuffisante
3. `add_decrease_effect` modifie le fluent numérique a chaque action — un solveur numérique (ex. ENHSP) en tiendrait compte dans sa recherche (Fast Downward, moteur classique, ne gère pas les fluents numériques)
4. Le plan attendu minimise les mouvements (2 actions pour A→B→C), consommant 20/50 unites

### Exercice 2 — Ajouter une action de recharge au robot

Clonez le problème dans `AddChargingStation` et faites atteindre D au robot suivant le trajet A–B–C–D. Fixez la charge initiale à 20 et conservez un coût de 10 par déplacement. Les trois déplacements demandent 30 unités, de sorte qu'une recharge est nécessaire dans le modèle de l'exercice.

- Ajoutez le lieu D, l'arête C→D et un but qui place le robot en D.
- Déclarez le fluent `charging_station`. La station est en B. L'action `recharge(l)` exige `at(l)` et `charging_station(l)`.
- L'action attribue une charge de 50. Avec Unified Planning, `add_effect(battery(), 50)` affecte ce niveau, ce qui évite de dépasser la capacité par des recharges répétées.
- Choisissez un moteur capable de traiter les fluents numériques. Vérifiez les actions et leur bilan d'énergie.

In [14]:
#nullable enable
// Exercice 2 : ajouter une station de recharge à B et atteindre un quatrième lieu.
PyObject? AddChargingStation(PyObject originalProblem)
{
    // TODO étudiant : cloner le problème et ajouter charging_station, l'action recharge et le lieu D.
    return null;
}
var possibleRecharge = AddChargingStation((PyObject)batteryProblem);
Console.WriteLine(possibleRecharge is null ? "Exercice de recharge à compléter." : "Problème de recharge construit.");

Exercice de recharge à compléter.


### Exemple guide 2 : Optimisation -- minimiser le coût total d'un plan

Jusqu'ici, les planificateurs ont resolu des problèmes de **satisfaction** :
trouver UN plan realisable (une sequence d'actions menant au but), comme le
robot qui atteint sa destination avec assez de batterie. C'est l'équivalent
planification de `Solver().check() == sat` en SMT.

La capacité distinctive d'un **planificateur d'optimisation** est la **quality
metric** : parmi tous les plans realisables, trouver celui qui **minimise un
coût** (ou maximise une récompense). C'est le coeur metier de la planification
**logistique** (tournees de livraison minimisant le kilometrage), **routing**
(réseaux), **ordonnancement** (makespan minimal). unified-planning l'exprime
via `problem.add_quality_metric(MinimizeActionCosts(...))`, et un moteur
**optimal** (ex. `fast-downward-opt`) **certifie** qu'aucun plan moins cher
n'existe -- la ou un solveur **satisficing** (`fast-downward`) se contente du
premier plan trouve, sans garantie de coût.

**Demonstration** : une livraison de A vers D sur un graphe a **coûts
heterogenes**. Le saut direct `A -> D` coute cher (10) ; le detour
`A -> B -> C -> D` coûte 3 unités (1 + 1 + 1), soit 7 unités de moins. Le plan le plus court
(1 action) **n'est pas** le moins cher -- c'est précisément ce qui rend
l'optimisation discriminante (un cas uniforme la rendrait triviale).

In [15]:
#nullable enable
dynamic delivery;
dynamic edgeCost;
using (Py.GIL())
{
    dynamic loc = up.UserType("DeliveryLocation");
    dynamic deliveryAt = up.Fluent("at", up.BoolType(), l: loc);
    dynamic deliveryAdjacent = up.Fluent("adjacent", up.BoolType(), l1: loc, l2: loc);
    edgeCost = up.Fluent("edge_cost", up.IntType(), l1: loc, l2: loc);
    dynamic deliveryMove = up.InstantaneousAction("move", l_from: loc, l_to: loc);
    dynamic from = deliveryMove.parameter("l_from");
    dynamic to = deliveryMove.parameter("l_to");
    deliveryMove.add_precondition(deliveryAt(from));
    deliveryMove.add_precondition(deliveryAdjacent(from, to));
    deliveryMove.add_effect(deliveryAt(from), false);
    deliveryMove.add_effect(deliveryAt(to), true);
    delivery = up.Problem("delivery_min_cost");
    delivery.add_fluent(deliveryAt, default_initial_value: false);
    delivery.add_fluent(deliveryAdjacent, default_initial_value: false);
    delivery.add_fluent(edgeCost, default_initial_value: 0);
    delivery.add_action(deliveryMove);
    var places = new[] { "A", "B", "C", "D" }.ToDictionary(label => label, label => (dynamic)up.Object(label, loc));
    foreach (dynamic place in places.Values)
        delivery.add_object(place);
    delivery.set_initial_value(deliveryAt(places["A"]), true);

    // Les deux routes sont accessibles dans les deux sens. Leur coût aller fait diverger longueur et optimum.
    foreach (var (a, b, cost) in new[] { ("A", "B", 1), ("B", "C", 1), ("C", "D", 1), ("A", "D", 10) })
    {
        delivery.set_initial_value(deliveryAdjacent(places[a], places[b]), true);
        delivery.set_initial_value(deliveryAdjacent(places[b], places[a]), true);
        delivery.set_initial_value(edgeCost(places[a], places[b]), cost);
    }
    delivery.add_goal(deliveryAt(places["D"]));
    using var actionCosts = new PyDict();
    actionCosts.SetItem((PyObject)deliveryMove, (PyObject)edgeCost(from, to));
    delivery.add_quality_metric(up.MinimizeActionCosts(costs: actionCosts));
}
int MovementCost(PlanningRun run)
{
    using (Py.GIL())
    {
        if (run.Plan is null)
            throw new InvalidOperationException("Un coût de trajectoire exige un plan.");
        dynamic steps = run.Plan.GetAttr("actions");
        int total = 0;
        for (int i = 0; i < run.Actions.Length; i++)
        {
            dynamic arguments = steps[i].actual_parameters;
            total += ((PyObject)delivery.initial_value(edgeCost(arguments[0], arguments[1])).constant_value()).As<int>();
        }
        return total;
    }
}
var optimum = Solve((PyObject)delivery, "fast-downward-opt");
var satisfying = Solve((PyObject)delivery, "fast-downward");
ShowPlan(optimum);
ShowPlan(satisfying);
int minimumCost = MovementCost(optimum);
int foundCost = MovementCost(satisfying);
Console.WriteLine($"Comparaison réelle : {minimumCost} pour l’optimiseur, {foundCost} pour le solveur de satisfaction.");
Console.WriteLine($"Écart de coût mesuré : {foundCost - minimumCost}.");

fast-downward-opt | SOLVED_OPTIMALLY | 0.249 s | 3 actions


  1. move(A, B)


  2. move(B, C)


  3. move(C, D)


fast-downward | SOLVED_SATISFICING | 0.219 s | 1 actions


  1. move(A, D)


Comparaison réelle : 3 pour l’optimiseur, 10 pour le solveur de satisfaction.


Écart de coût mesuré : 7.


### Interpretation : Optimisation vs satisfaction en planification

**Sortie obtenue** : `fast-downward-opt` (optimal) trouve le detour
`A -> B -> C -> D` (3 actions, coût **3**) et le **certifie minimal** ;
`fast-downward` (satisficing) se contente du saut direct `A -> D` (1 action,
coût **10**). L'optimal bat le satisficing de **7 unites** -- **avec plus
d'actions**. C'est la signature d'un problème d'optimisation non-trivial : le
coût distingue des plans de même validite, et seul le moteur optimal explore
l'espace assez pour trouver le moins cher.

**Pourquoi c'est la capaute signature** :
- Sans quality metric, un planificateur se réduit a un **chercheur de plan**
  (BFS / GBFS sur le graphe d'états) -- l'optimalité n'est pas du tout
  recherchee.
- Avec `MinimizeActionCosts`, le moteur devient un **optimiseur combinatoire**
  (A* sur le graphe pondere, heuristique admissible, garantie d'optimalité).
- C'est exactement ce qui distingue un outils industriel (IPC optimal track)
  d'une demo pédagogique de recherche.

**Miroir** de la capabilite signature de CP-SAT (`model.Maximize`) et de Z3
(`Optimize().maximize()`) : dans les trois familles (planification, CP, SMT),
l'optimization est le saut qualitatif au-dela de la satisfaction. Voir
`Sudoku-10-ORTools-Python` (#7588) et `Sudoku-12-Z3-Python` (#7589).

> **Note technique** : `MinimizeActionCosts` mappe chaque action a une
> expression de coût (ici `edge_cost(l_from, l_to)`, un fluent numérique
> initialise par arete). D'autres metrics : `MinimizeMakespan` (planification
> temporelle -- durée totale), `MinimizeExpressionOnFinalState` (minimiser
> une ressource residuelle), `MaximizeExpressionOnFinalState` (maximiser un
> gain).

***

## 5. Integration PDDL

unified-planning peut lire et ecrire du PDDL, permettant l'interoperabilite avec l'ecosysteme existant.

### 5.1 Export vers PDDL

La conversion vers PDDL permet d'utiliser des outils qui ne supportent que ce format.

In [16]:
#nullable enable
string domainPddl;
string problemPddl;
using (Py.GIL())
{
    // Le writer sérialise le projet déjà résolu, avec les fluents et l'action construits.
    dynamic writer = pddl.PDDLWriter(navigation);
    domainPddl = writer.get_domain().ToString();
    problemPddl = writer.get_problem().ToString();
}
Console.WriteLine("Domaine produit par PDDLWriter");
Console.WriteLine(domainPddl);

Domaine produit par PDDLWriter


(define (domain robot_navigation-domain)
 (:requirements :strips :typing)
 (:types robot location)
 (:predicates 
             (robot_at ?r - robot ?l - location)
             (connected ?l1 - location ?l2 - location)
             (visited ?l - location)
 )
 (:action move
  :parameters ( ?r - robot ?l_from - location ?l_to - location)
  :precondition (and (connected ?l_from ?l_to) (robot_at ?r ?l_from))
  :effect (and (not (robot_at ?r ?l_from)) (robot_at ?r ?l_to) (visited ?l_to)))
)



Generation et affichage du fichier PDDL du problème (problem.pddl) exporté, contenant les objets, l'état initial et le but sous forme textuelle standardisee.

In [17]:
#nullable enable
// Ce fichier porte l'instance plutôt que les règles génériques du domaine.
Console.WriteLine("Problème produit par PDDLWriter");
Console.WriteLine(problemPddl);

Problème produit par PDDLWriter


(define (problem robot_navigation-problem)
 (:domain robot_navigation-domain)
 (:objects
   robot1 - robot
   l0 l1 l2 l3 l4 l5 l6 l7 l8 l9 l10 l11 l12 l13 l14 - location
 )
 (:init
              (connected l0 l1)
              (connected l1 l0)
              (connected l1 l2)
              (connected l2 l1)
              (connected l2 l4)
              (connected l4 l2)
              (connected l4 l7)
              (connected l7 l4)
              (connected l7 l10)
              (connected l10 l7)
              (connected l10 l13)
              (connected l13 l10)
              (connected l10 l11)
              (connected l11 l10)
              (connected l13 l11)
              (connected l11 l13)
              (connected l1 l3)
              (connected l3 l1)
              (connected l4 l5)
              (connected l5 l4)
              (connected l2 l6)
              (connected l6 l2)
              (connected l7 l8)
              (connected l8 l7)
              (connected l2 l9)
    

### 5.2 Import depuis PDDL

La lecture de fichiers PDDL permet de travailler avec des domaines existants.

In [18]:
#nullable enable
var pddlDirectory = Directory.CreateTempSubdirectory("coursia-planners11-");
string domainFile = Path.Combine(pddlDirectory.FullName, "domain.pddl");
string problemFile = Path.Combine(pddlDirectory.FullName, "problem.pddl");
File.WriteAllText(domainFile, domainPddl);
File.WriteAllText(problemFile, problemPddl);
try
{
    using (Py.GIL())
    {
        dynamic reloaded = pddl.PDDLReader().parse_problem(domainFile, problemFile);
        int actions = (int)((PyObject)reloaded.actions).Length();
        int objects = (int)((PyObject)reloaded.all_objects).Length();
        int goals = (int)((PyObject)reloaded.goals).Length();
        Console.WriteLine($"Problème relu : {reloaded.name} | {actions} action | {objects} objets | {goals} but.");
        Console.WriteLine($"Identité des nombres d'actions et d'objets : {actions == ((PyObject)navigation.actions).Length() && objects == ((PyObject)navigation.all_objects).Length()}.");
    }
}
finally
{
    // Le carnet ne détruit que ses deux fichiers temporaires privés.
    File.Delete(domainFile);
    File.Delete(problemFile);
    Directory.Delete(pddlDirectory.FullName);
}

Problème relu : robot_navigation-problem | 1 action | 16 objets | 1 but.


Identité des nombres d'actions et d'objets : True.


### Interpretation : Integration PDDL

**Quand utiliser unified-planning vs PDDL direct ?**

| Situation | Approche recommandee |
|-----------|---------------------|
| Prototypage rapide | unified-planning (API Python) |
| Integration avec code Python | unified-planning |
| Utilisation de domaines existants | Import PDDL -> unified-planning |
| Soumission a competitions IPC | Export PDDL depuis unified-planning |
| Debug pas a pas | unified-planning (introspection Python) |
| Performance maximale | PDDL direct avec Fast Downward |

**Points clés** :
1. PDDLWriter/Reader assure la compatibilite avec l'ecosysteme PDDL
2. L'API Python est plus flexible pour la generation dynamique de problèmes
3. Certains commentaires et structures peuvent ne pas survivre a la conversion aller-retour

***

## 6. Bonnes pratiques

### 6.1 Choix du solveur

Le type du problème indique quelles fonctionnalités le moteur doit prendre en charge. Ce choix donne un candidat à vérifier dans la liste des moteurs disponibles. Tamer et ENHSP nécessitent leurs propres plugins pour les problèmes temporels et numériques.

```csharp
string SelectSolver(PyObject problem)
{
    using (Py.GIL())
    {
        dynamic kind = problem.GetAttr("kind");
        if (kind.has_continuous_time() || kind.has_discrete_time())
            return "tamer";
        if (kind.has_numeric_fluents())
            return "enhsp";
        if (kind.has_actions_cost())
            return "fast-downward-opt";
        return "pyperplan";
    }
}
```

### 6.2 Gestion des erreurs

Le statut indique si un plan existe, si une impossibilité est prouvée ou si une autre tentative peut être utile. L'exemple suivant s'appuie sur le résultat structuré de `Solve`.

```csharp
var answer = Solve((PyObject)navigation, "pyperplan", 30);
switch (answer.Status)
{
    case "SOLVED_SATISFICING":
    case "SOLVED_OPTIMALLY":
        ShowPlan(answer);
        break;
    case "UNSOLVABLE_PROVEN":
        Console.WriteLine("Impossibilité prouvée.");
        break;
    case "UNSOLVABLE_INCOMPLETELY":
    case "TIMEOUT":
        Console.WriteLine("La recherche peut être reprise avec un autre moteur.");
        break;
    default:
        Console.WriteLine($"Statut du moteur {answer.Status}.");
        break;
}
```

In [19]:
#nullable enable
PlanningRun? SolveInOrder(PyObject instance, string preferred, string alternative, int timeout = 30)
{
    // Un plan termine la recherche. Une impossibilité prouvée termine également le diagnostic.
    foreach (string engine in new[] { preferred, alternative })
    {
        try
        {
            var attempt = Solve(instance, engine, timeout);
            Console.WriteLine($"{engine} | {attempt.Status} | {attempt.Actions.Length} actions.");
            if (attempt.Solved || attempt.Status == "UNSOLVABLE_PROVEN")
                return attempt;
        }
        catch (PythonException ex)
        {
            Console.WriteLine($"{engine} | moteur non résolu : {ex.GetType().Name}.");
        }
    }
    return null;
}
var robustRun = SolveInOrder((PyObject)navigation, "pyperplan", "fast-downward");
Console.WriteLine(robustRun is not null
    ? $"Moteur retenu : {robustRun.Engine} | {robustRun.Status}."
    : "Les deux tentatives ont terminé sans plan ni preuve d'impossibilité.");

pyperplan | SOLVED_SATISFICING | 6 actions.


Moteur retenu : pyperplan | SOLVED_SATISFICING.


### Interprétation — Résolution robuste avec gestion d'erreurs

La sortie affiche `pyperplan | SOLVED_SATISFICING | 6 actions.`, puis `Moteur retenu : pyperplan | SOLVED_SATISFICING.` Le premier moteur suffit sur cette instance.

`SolveInOrder` renvoie le résultat quand le premier moteur résout le problème ou prouve son impossibilité. Si la recherche expire, se termine sans preuve ou déclenche une exception Python, la boucle peut essayer le moteur suivant. La comparaison précédente a aussi exécuté Fast Downward sur ce même problème et obtenu six actions.

***

## À retenir

### 7.1 Concepts appris

| Concept | Description |
|---------|-------------|
| **unified-planning** | Interface Python unifiee pour multiples planificateurs |
| **Fluent** | Prédicat ou fonction variable selon l'état |
| **InstantaneousAction** | Action avec préconditions/effets instantanes |
| **DurativeAction** | Action avec durée et effets temporises |
| **OneshotPlanner** | Planificateur one-shot (un problème, un plan) |
| **PDDLWriter/Reader** | Conversion entre API Python et PDDL |

### 7.2 Points clés a retenir

1. **Abstraction** : unified-planning permet d'ecrire le problème une fois, de le résoudre avec n'importe quel solveur

2. **Comparaison** : Facilite le benchmarking de différents planificateurs

3. **Extensibilite** : Supporte PDDL classique, temporel, et numérique

4. **Interoperabilite** : Import/export PDDL pour compatibilite avec l'ecosysteme existant

5. **Choix du solveur** : Adapter le solveur au type de problème (classique vs temporel vs numérique)

### 7.3 Quand utiliser unified-planning ?

| Cas d'usage | Recommandation |
|-------------|----------------|
| Recherche / Prototypage | unified-planning (flexibilite) |
| Production (performance) | PDDL direct + Fast Downward |
| Integration Python | unified-planning |
| Benchmark multi-solveurs | unified-planning |
| Domaines IPC existants | Import PDDL -> unified-planning |

***

## Ressources

- [unified-planning GitHub](https://github.com/aiplan4eu/unified-planning)
- [Documentation officielle](https://unified-planning.readthedocs.io/)
- [AIPlan4EU Project](https://www.aiplan4eu-project.eu/)
- [Tutoriels](https://unified-planning.readthedocs.io/en/latest/tutorials/index.html)

***

**Notebook suivant** : [Planners-12-LOOP-Python](Planners-12-LOOP-Python.ipynb) - Learning to Plan avec modèles neuronaux

## Resume et perspectives

Ce notebook a presente **unified-planning**, une bibliotheque Python qui unifie l'acces a de multiples planificateurs sous une API coherente. Nous avons défini les problèmes dans le C# en appelant l’API Unified Planning via Python.NET (types, fluents, actions, préconditions, effets), resolu ces problèmes avec différents solveurs (Fast Downward, Pyperplan), et compare leurs performances en termes de temps de resolution et qualité de plan. Les fonctionnalites avancees -- actions duratives, fluents numériques, contraintes de durée et de ressources -- montrent la richesse expressive de l'API au-dela du simple modèle STRIPS. L'integration PDDL via PDDLWriter et PDDLReader assure la compatibilite avec l'ecosysteme de planification existant.

L'intérêt principal d'unified-planning reside dans l'abstraction qu'il offre : un problème défini une seule fois peut etre resolu par n'importe quel solveur compatible, ce qui facilite considerablement le benchmarking et la sélection du solveur optimal pour un domaine donne. La gestion robuste des erreurs (fallback entre solveurs, gestion des timeouts, statuts detailles) est essentielle en production, ou la fiabilite prime sur la performance pure. Le choix du solveur doit etre guide par les caractéristiques du problème : planification classique pour Pyperplan ou Fast Downward, fluents numériques pour ENHSP, planification temporelle pour LPG ou Tamer.

Le notebook suivant, [Planners-12-LOOP-Python](Planners-12-LOOP-Python.ipynb), explore le paradigme **Learning to Plan**, ou des réseaux de neurones apprennent des heuristiques de planification a partir de données, complementant les approches symboliques etudiees ici.

***

### Exercice 3 — Comparer deux solveurs sur un problème original

Utilisez `unified-planning` pour résoudre **votre propre problème** avec
**deux solveurs différents** et comparez leurs performances.

### Contraintes
- Problème original (pas Block World ni Transport)
- Tester avec **au moins 2 solveurs** (ex: `pyperplan` + `tamer` ou `enhsp`)
- Mesurer et comparer le **temps de resolution** et la **longueur du plan**

> **Étapes :**
1. Définir un problème de planification original avec `unified_planning`
2. Résoudre avec `OneshotPlanner` pour chaque solveur
3. Comparer les résultats : statut, longueur du plan, temps
4. (Bonus) Utiliser `AnytimePlanner` pour obtenir des plans sous-optimaux progressifs

In [20]:
#nullable enable
// Exercice 3 : modéliser un problème original que deux moteurs peuvent résoudre.
PyObject? OriginalProblem()
{
    // TODO étudiant : définir les objets, actions, état initial et but.
    return null;
}
var original = OriginalProblem();
if (original is null)
    Console.WriteLine("Exercice de comparaison à compléter.");
else
    ComparisonTable(enginesToCompare.Select(engine => Solve(original, engine, 30)).ToArray());

Exercice de comparaison à compléter.


## Vérifiez votre compréhension

1. Que reste-t-il identique quand on passe de Pyperplan à Fast Downward dans le labyrinthe ?
2. Quel est le plan le moins cher sur le graphe de livraison ?
3. Qu'établit le statut du solveur dans l'exemple numérique avec batterie ?

<details>
<summary>Éléments de réponse</summary>

1. Le modèle et son objectif sont identiques. Les deux plans ont six actions.
2. L'optimiseur choisit A–B–C–D, trois actions de coût total 3. Le solveur de satisfaction choisit le saut direct de coût 10.
3. Fast Downward rend `INTERNAL_ERROR` sur ces fluents numériques. Le calcul manuel des deux actions sert à comprendre le modèle proposé.

</details>

## Pour aller plus loin

Le [carnet sur la qualité des plans](../03-Advanced/Planners-13-Mesure-Qualite-Plans-CSharp.ipynb) compare les heuristiques de Fast Downward sur une même tour.